# Comparing the Models: LSTM and XGBoost
This is my comparison notebook for the initial two models I wanted to try, LSTM and XGBoost. By the end of these comparisons, I'll have selected a model to move forward with tuning.

## Imports and Prediction Loading

In [8]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')
 
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
 
print("Imports successful")

# Prediction loading
print("\nLoading model predictions...")
 
# XGBoost: trained on all 117K pitches
xgb_pred = pd.read_csv('xgboost/xgboost_predictions.csv')
print(f"XGBoost predictions: {len(xgb_pred)} individual pitches")
print(f"  (trained on feature-engineered data, all pitches)")
 
# LSTM: trained on 98K sequences  
lstm_pred = pd.read_csv('lstm/lstm_seq_predictions.csv')
print(f"LSTM predictions: {len(lstm_pred)} sequences")
print(f"  (trained on temporal sequences: last 5 pitches → predict next)")
 
print(f"\nNote: Different test sets!")
print(f"  XGBoost: 117,758 individual pitch predictions")
print(f"  LSTM: 98,161 sequence-based predictions")
print(f"  -> Can't do 1:1 comparison, but can compare strategies")
 
# Load pitch encoder and metadata
with open('../feature-engineering/pitch_type_encoder.pkl', 'rb') as f:
    pitch_encoder = pickle.load(f)
 
with open('lstm/seq_metadata.pkl', 'rb') as f:
    seq_metadata = pickle.load(f)
 
pitch_types = pitch_encoder.classes_

Imports successful

Loading model predictions...
XGBoost predictions: 117758 individual pitches
  (trained on feature-engineered data, all pitches)
LSTM predictions: 38056 sequences
  (trained on temporal sequences: last 5 pitches → predict next)

Note: Different test sets!
  XGBoost: 117,758 individual pitch predictions
  LSTM: 98,161 sequence-based predictions
  -> Can't do 1:1 comparison, but can compare strategies


## Overall Accuracy Comparison

Since the two models are analyzing different inputs, we can't make a direct comparison, but we can still use the numbers to help us determine which model performs better.

In [9]:
# XGBoost: all pitches
y_actual_xgb = xgb_pred['actual_pitch_idx'].values
y_pred_xgb = xgb_pred['predicted_pitch_idx'].values
xgb_acc = accuracy_score(y_actual_xgb, y_pred_xgb)
 
# LSTM: sequences
y_actual_lstm = lstm_pred['actual_pitch_idx'].values
y_pred_lstm = lstm_pred['predicted_pitch_idx'].values
lstm_acc = accuracy_score(y_actual_lstm, y_pred_lstm)
 
# Baselines (from their respective test sets)
xgb_baseline = (y_actual_xgb == 0).sum() / len(y_actual_xgb)  # Assuming 0 = FF
lstm_baseline = (y_actual_lstm == 0).sum() / len(y_actual_lstm)
 
print(f"\nXGBoost (All Pitches):")
print(f"  Test set size: {len(xgb_pred):,} individual pitches")
print(f"  Baseline: {xgb_baseline:.1%}")
print(f"  Accuracy: {xgb_acc:.1%}")
print(f"  Improvement: +{(xgb_acc - xgb_baseline)*100:.1f} percentage points")
 
print(f"\nLSTM (Sequences):")
print(f"  Test set size: {len(lstm_pred):,} sequences (last {seq_metadata['seq_length']} pitches -> predict next)")
print(f"  Baseline: {lstm_baseline:.1%}")
print(f"  Accuracy: {lstm_acc:.1%}")
print(f"  Improvement: +{(lstm_acc - lstm_baseline)*100:.1f} percentage points")

if xgb_acc > lstm_acc:
    winner = "XGBoost"
    winner_margin = (xgb_acc - lstm_acc) * 100
    print(f"XGBoost wins on its test set: {xgb_acc:.1%} vs {lstm_acc:.1%}")
    print(f"   BUT: Different test sets, so margin ({winner_margin:.1f}pp) is not directly comparable")
else:
    winner = "LSTM"
    winner_margin = (lstm_acc - xgb_acc) * 100
    print(f"LSTM wins on its test set: {lstm_acc:.1%} vs {xgb_acc:.1%}")
    print(f"   BUT: Different test sets, so margin ({winner_margin:.1f}pp) is not directly comparable")


XGBoost (All Pitches):
  Test set size: 117,758 individual pitches
  Baseline: 11.8%
  Accuracy: 43.6%
  Improvement: +31.7 percentage points

LSTM (Sequences):
  Test set size: 38,056 sequences (last 5 pitches -> predict next)
  Baseline: 13.6%
  Accuracy: 34.5%
  Improvement: +20.9 percentage points
XGBoost wins on its test set: 43.6% vs 34.5%
   BUT: Different test sets, so margin (9.1pp) is not directly comparable


## Per-pitch performance analysis

In [10]:
print("\nXGBoost per-pitch accuracy (117K pitches):")
print(f"{'Pitch':<5} {'Accuracy':<10} {'Count':<8}")
print("-" * 25)
for i, pitch in enumerate(pitch_types):
    mask = y_actual_xgb == i
    if mask.sum() > 0:
        acc = accuracy_score(y_actual_xgb[mask], y_pred_xgb[mask])
        print(f"{pitch:<5} {acc:.1%}         {mask.sum():<8}")
 
print("\nLSTM per-pitch accuracy (98K sequences):")
print(f"{'Pitch':<5} {'Accuracy':<10} {'Count':<8}")
print("-" * 25)
for i, pitch in enumerate(pitch_types):
    mask = y_actual_lstm == i
    if mask.sum() > 0:
        acc = accuracy_score(y_actual_lstm[mask], y_pred_lstm[mask])
        print(f"{pitch:<5} {acc:.1%}         {mask.sum():<8}")


XGBoost per-pitch accuracy (117K pitches):
Pitch Accuracy   Count   
-------------------------
CH    32.0%         13948   
CS    0.0%         6       
CU    7.6%         5394    
EP    95.5%         133     
FA    7.1%         14      
FC    25.0%         8671    
FF    66.8%         36463   
FS    26.9%         2866    
KC    0.1%         3511    
PO    0.0%         4       
SI    53.7%         20704   
SL    28.9%         16164   
ST    32.8%         9693    
SV    0.0%         185     
UN    0.0%         2       

LSTM per-pitch accuracy (98K sequences):
Pitch Accuracy   Count   
-------------------------
CH    0.0%         5167    
CU    0.0%         1830    
EP    0.0%         2       
FA    0.0%         2       
FC    0.1%         2954    
FF    93.0%         11893   
FS    2.9%         972     
KC    0.0%         1702    
PO    0.0%         2       
SI    33.4%         5787    
SL    1.1%         4991    
ST    1.2%         2689    
SV    0.0%         65      


## Model Philosophy Comparison

### XGBoost
- Trains on individual pitches
- Learns feature interactions (pitch arsenals, counts, handedness, etc.)
- No temporal memory (treats each pitch independently)
- Good for overall pitch prediction and interpretability

### LSTM:
- Clearly trolling me, as it only predicts 4S fastballs
- Trains on sequences
- Learns pitch history and has temporal memory
- Has less data to work with due to cleaning some of the sequences out (only took 4+ pitch ABs)
- Good for at-bat dynamics and sequence understanding

## Testing History
I want to try some different hyperparameters and whatnot, so I'll log them here:

### XGBoost
- 0: n_estimators=200, max_depth=7, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=1, tree_method='hist'
    - 47.2% train acc, 43.5% test acc
- **1**: 500 trees instead of 200, learning rate 0.05, subsample=0.8
    - 48.1% train acc, 43.6% test acc
- 2: Added min_child_weight=5, reg_alpha=0.1, reg_lambda=5
    - 47.7% train acc, 43.6% test acc
### LSTM
- 0: epochs=30, batch_size=512, validation_split=0.2, early stopping=3, verbose=1, pitch sequences were 4+
    - 36% train acc, 35.6% test acc
    - Converged at epoch 25
- **1**: Hyperparameters untouched, pitch sequences changed to 3+
    - 36% train acc, 35.4% test acc
    - Converged at epoch 18
- 2: Hyperparameters untouched, pitch sequences changed to 5+
    - 35.2% train acc, 34.5% test acc
    - Converged at epoch

## Export

In [11]:
summary_df = pd.DataFrame({
    'model': ['XGBoost (all pitches)', 'LSTM (sequences)'],
    'test_set_size': [len(xgb_pred), len(lstm_pred)],
    'test_set_description': ['117K individual pitches', '98K sequences (last 5→predict 6)'],
    'accuracy': [xgb_acc, lstm_acc],
    'baseline': [xgb_baseline, lstm_baseline],
    'improvement': [(xgb_acc - xgb_baseline), (lstm_acc - lstm_baseline)],
})
 
summary_df.to_csv('phase1_comparison_summary.csv', index=False)
print("Saved lstm_xgboost_comparison_summary.csv")
print("\n" + summary_df.to_string(index=False))

Saved lstm_xgboost_comparison_summary.csv

                model  test_set_size             test_set_description  accuracy  baseline  improvement
XGBoost (all pitches)         117758          117K individual pitches  0.435639  0.118446     0.317193
     LSTM (sequences)          38056 98K sequences (last 5→predict 6)  0.344545  0.135774     0.208771


## Conclusion of Model Comparison
We're going with XGBoost. LSTM is underwhelming on pretty much every statistic, largely in part due to the fact that it can only predict 4S fastballs.